<a href="https://colab.research.google.com/github/HillaryDrugs/li7/blob/main/Distel_Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


!pip -q install -U "transformers>=4.41.0" accelerate datasets jiwer soundfile huggingface_hub torch

import re, io
import numpy as np
import torch
import soundfile as sf

from datasets import load_dataset, Audio
from jiwer import process_words
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# -----------------------------
# 1) Config
# -----------------------------
DATASET_ID   = "NightPrince/MasriSpeech-Full"
SPLIT        = "validation"
NUM_SAMPLES  = 500
TARGET_SR    = 16000

# Arabic Distil-Whisper checkpoint
MODEL_ID = "distil-whisper-ar/distil-whisper-16-16"

device = "cuda" if torch.cuda.is_available() else "cpu"
# Use fp16 ONLY on GPU; use fp32 on CPU
model_dtype = torch.float16 if device == "cuda" else torch.float32
input_dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Model:", MODEL_ID)
print("Model dtype:", model_dtype)

# -----------------------------
# 2) Arabic normalization
# -----------------------------
def normalize_ar(text):
    if not text: return ""
    text = text.lower()
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[ؤئ]', 'ء', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'[^\u0621-\u064A\s\d]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# -----------------------------
# 3) WER breakdown (your format)
# -----------------------------
def wer_breakdown(refs, hyps):
    total_words = total_del = total_sub = perfect = 0

    for r, h in zip(refs, hyps):
        m = process_words(r, h)
        n = m.hits + m.substitutions + m.deletions
        total_words += n
        total_del += m.deletions
        total_sub += m.substitutions
        if (m.substitutions + m.deletions + m.insertions) == 0:
            perfect += 1

    print("\n========================")
    print("FINAL RESULTS")
    print("========================")
    print(f"Perfect WER: {1 - perfect/len(refs):.1f}")
    print(f"Deletion WER: {total_del/total_words:.1f}")
    print(f"Typo WER: {total_sub/total_words:.1f}")

# -----------------------------
# 4) Load dataset WITHOUT decoding (avoids torchcodec)
# -----------------------------
print("Loading dataset (decode=False)...")
ds = load_dataset(DATASET_ID, split=SPLIT)
ds = ds.cast_column("audio", Audio(decode=False))
ds = ds.select(range(min(NUM_SAMPLES, len(ds))))
print("Samples used:", len(ds))

# -----------------------------
# 5) Audio loader from bytes + simple resample (no librosa/torchaudio)
# -----------------------------
def resample_linear(wav, sr, target_sr=16000):
    if sr == target_sr:
        return wav.astype(np.float32)
    x_old = np.linspace(0, 1, num=len(wav), endpoint=False)
    new_len = int(len(wav) * target_sr / sr)
    x_new = np.linspace(0, 1, num=new_len, endpoint=False)
    return np.interp(x_new, x_old, wav).astype(np.float32)

def load_audio_np(audio_obj):
    wav, sr = sf.read(io.BytesIO(audio_obj["bytes"]))
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    wav = resample_linear(wav, sr, TARGET_SR)
    return wav

# -----------------------------
# 6) Load Distil-Whisper model
# -----------------------------
print("Loading processor/model...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    low_cpu_mem_usage=True
).to(device)
model.eval()
print("✅ Loaded.")

# Try to force Arabic transcription prompt (if supported)
forced_ids = None
try:
    forced_ids = processor.get_decoder_prompt_ids(language="ar", task="transcribe")
except Exception:
    forced_ids = None

# -----------------------------
# 7) Run evaluation
# -----------------------------
refs, hyps = [], []
print("\nRunning Distil-Whisper ASR...")

with torch.no_grad():
    for i, ex in enumerate(ds):
        ref = normalize_ar(ex["transcription"])
        if not ref:
            continue

        wav = load_audio_np(ex["audio"])

        inputs = processor(wav, sampling_rate=TARGET_SR, return_tensors="pt")
        # Move to device + match dtype (THIS FIXES YOUR ERROR)
        for k in inputs:
            inputs[k] = inputs[k].to(device)
            if inputs[k].dtype == torch.float32 and input_dtype == torch.float16:
                inputs[k] = inputs[k].to(torch.float16)

        gen_kwargs = dict(max_new_tokens=128)
        if forced_ids is not None:
            gen_kwargs["forced_decoder_ids"] = forced_ids

        generated_ids = model.generate(**inputs, **gen_kwargs)
        hyp = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        hyp = normalize_ar(hyp)

        refs.append(ref)
        hyps.append(hyp)

        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(ds)}")

wer_breakdown(refs, hyps)
